# Quarkyonic Sound Speed $v_s^2$

This notebook computes the zero-temperature sound speed for the quarkyonic construction.
It reuses the already-working ground-state fit for `a`, `b`, and `K_0`, but it does **not**
use the hadronic-only sound-speed workflow.


## Mathematical Setup

At fixed baryon density `n_B`, the quarkyonic model is obtained by minimizing the total energy density
with respect to the quark fraction `f_Q = n_Q / n_B`.

For each trial `f_Q`, we define

$$
n_Q = f_Q n_B,
\qquad
n_N = n_B - n_Q.
$$

The quark shell momentum `k_{bu}` is determined from the quark density, while the nucleon Fermi momentum
`k_F` is reconstructed from the excluded-volume nucleon density. The energy density is then written as

$$
\epsilon(n_B, f_Q) = \epsilon_{N,\mathrm{shell}}(n_N, k_{bu}, k_F) + \epsilon_Q(k_{bu}).
$$

At every `n_B`, the physical branch is the minimum-energy solution,

$$
\epsilon(n_B) = \min_{f_Q} \epsilon(n_B, f_Q).
$$

After that minimum-energy branch is built, the thermodynamics are reconstructed numerically as

$$
\mu_B = \frac{d\epsilon}{dn_B},
\qquad
P = n_B \mu_B - \epsilon,
\qquad
v_s^2 = \frac{dP/dn_B}{d\epsilon/dn_B}.
$$

A rapid change in the minimizing quark fraction can produce a sharp kink in `\epsilon(n_B)`, and that
shows up as a peak in `v_s^2`. For that reason, the notebook also plots the optimal quark fraction branch
`f_Q(n_B)` as a diagnostic.

All derivatives are evaluated with local finite differences from the file
`quarkyonic_sound_speed_vs2/src/quarkyonic_sound_speed_vs2/utils/numerics.py`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
project_dir = None

for candidate in [cwd, cwd.parent, cwd / "quarkyonic_sound_speed_vs2"]:
    if (candidate / "src" / "quarkyonic_sound_speed_vs2").exists():
        project_dir = candidate
        break

if project_dir is None:
    raise RuntimeError("Could not locate quarkyonic_sound_speed_vs2/src.")

ground_state_src = project_dir.parent / "ground_state_ab_k0" / "src"
quarkyonic_src = project_dir / "src"
results_dir = project_dir / "results"
results_dir.mkdir(exist_ok=True)

for path in [str(ground_state_src), str(quarkyonic_src)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from quarkyonic_sound_speed_vs2 import QuarkyonicSettings, compute_selected_quarkyonic_curves
from quarkyonic_sound_speed_vs2.reporting import curve_to_records, summaries_to_records


In [ ]:
# Use QuarkyonicSettings() for the denser default run.
settings = QuarkyonicSettings.quick()
curves = compute_selected_quarkyonic_curves(settings=settings)


In [ ]:
summary_df = pd.DataFrame(summaries_to_records(curves))
curve_df = pd.DataFrame([row for curve in curves for row in curve_to_records(curve)])

summary_df.to_csv(results_dir / "quarkyonic_sound_speed_summary.csv", index=False)
curve_df.to_csv(results_dir / "quarkyonic_sound_speed_curves.csv", index=False)

display(summary_df[["model", "parameter_value", "a", "b", "K0", "vs2_max", "n_tr_over_n0"]])


In [ ]:
color_map = {
    "vdw": "black",
    "rks": "red",
    "pr": "blue",
    "dieterici": "green",
    "clausius": "magenta",
}

label_map = {
    "vdw": "VDW",
    "rks": "RKS",
    "pr": "PR",
    "dieterici": r"Dieterici, $\alpha = 5/3$",
    "clausius": r"Clausius, $c = 4.74\ \mathrm{fm}^3$",
}

linestyle_map = {
    "vdw": "-",
    "rks": ":",
    "pr": (0, (2, 2)),
    "dieterici": "-.",
    "clausius": (0, (4, 2, 1, 2)),
}

fig, ax = plt.subplots(figsize=(8, 4.8))

for curve in curves:
    ax.plot(
        curve.n_over_n0,
        curve.quark_fraction,
        color=color_map[curve.model],
        lw=2.0,
        ls=linestyle_map[curve.model],
        label=label_map[curve.model],
    )

ax.set_xlabel(r"$n$ ($n_0$)", fontsize=15)
ax.set_ylabel(r"$f_Q = n_Q/n_B$", fontsize=15)
ax.set_xlim(0.0, 5.0)
ax.set_ylim(0.0, 1.0)
ax.legend(frameon=False, fontsize=10, loc="upper left")
ax.grid(alpha=0.2)

fig.tight_layout()
fig.savefig(results_dir / "quarkyonic_fraction_selected.png", dpi=300)
plt.show()


In [ ]:
color_map = {
    "vdw": "black",
    "rks": "red",
    "pr": "blue",
    "dieterici": "green",
    "clausius": "magenta",
}

label_map = {
    "vdw": "VDW",
    "rks": "RKS",
    "pr": "PR",
    "dieterici": r"Dieterici, $\alpha = 5/3$",
    "clausius": r"Clausius, $c = 4.74\ \mathrm{fm}^3$",
}

linestyle_map = {
    "vdw": "-",
    "rks": ":",
    "pr": (0, (2, 2)),
    "dieterici": "-.",
    "clausius": (0, (4, 2, 1, 2)),
}

fig, ax = plt.subplots(figsize=(8, 6))

for curve in curves:
    ax.plot(
        curve.n_over_n0,
        curve.vs2,
        color=color_map[curve.model],
        lw=2.0,
        ls=linestyle_map[curve.model],
        label=label_map[curve.model],
    )

ax.axhline(1.0 / 3.0, color="gray", lw=1.5, ls=(0, (5, 5)))
ax.text(0.22, 1.0 / 3.0 + 0.02, r"$v_s^2 = 1/3$", color="dimgray", fontsize=13)

ax.set_xlabel(r"$n$ ($n_0$)", fontsize=15)
ax.set_ylabel(r"$v_s^2$", fontsize=15)
ax.set_xlim(0.0, 5.0)
y_max = max(1.0, 1.05 * max(curve.vs2_max for curve in curves))
ax.set_ylim(-0.08, y_max)
ax.legend(frameon=False, fontsize=11, loc="lower right")
ax.grid(alpha=0.2)

fig.tight_layout()
fig.savefig(results_dir / "quarkyonic_sound_speed_selected.png", dpi=300)
plt.show()
